In [2]:
import numpy as np
from WaveNewmark import *
from WaveNewmarkFractional import *
from scipy.integrate import simps
import matplotlib.pyplot as plt
from NewObservation import *
from petsc4py import PETSc
from scipy.linalg import sqrtm, inv, ldl
from mshr import *

In [3]:
# Define mesh:
# mesh = fn.Mesh('mesh_100.xml')
rectangle = Rectangle(fn.Point(-1., -1.), fn.Point(1., 1.))
circle = Circle(fn.Point(0, 0), 0.8, segments=50)
# Generate mesh:
mesh = get_mesh(rectangle, circle, 30)
V = fn.FunctionSpace(mesh, "CG", 1)
V.tabulate_dof_coordinates()
# Degree of freedom:
print('Degree of freedom: ', V.dim())

In [4]:
# Define subdomain and boundary:
subdomains = fn.MeshFunction("size_t", mesh, 2, mesh.domains())
boundaries = fn.MeshFunction('size_t', mesh, 1, mesh.domains())
for f in fn.facets(mesh):
    domains = []
    for c in fn.cells(f):
        domains.append(subdomains[c])
    domains = list(set(domains))
    if len(domains) > 1:
        boundaries[f] = 1
       
dS = fn.dS(subdomain_data=boundaries)
u_trial, u_test = fn.TrialFunction(V), fn.TestFunction(V)
B_matrix = fn.assemble(fn.inner(u_trial('+'), u_test('+')) * dS(1))

u_temp = init_vector(B_matrix, 0)
u_temp[:] = 1.
Bu = B_matrix * u_temp
targets = Bu[:][np.nonzero(Bu[:])]
support = np.nonzero(Bu[:])[0]
print('Number of observations: ', len(support))
print('Plot observations: ')
coordinates = V.tabulate_dof_coordinates()[support]
plt.plot(coordinates[:, 0], coordinates[:, 1], 'ro')
plt.axis('equal')
plt.xlim(-1, 1)
plt.ylim(-1, 1)
plt.show()

In [5]:
# At the same time, check with get_boundaries function:
def get_boundaries(V, boundaries):
    # Get boundary indices
    mesh = V.mesh()
    facet_indices = np.flatnonzero(boundaries.array() == 1)
    # Create facet to vertex connectivity
    mesh.init(mesh.topology().dim()-1, 0)  
    # Get mesh nodes
    f_to_c = mesh.topology()(mesh.topology().dim()-1, 0) 
    temp_indices = np.array([f_to_c(f) for f in facet_indices])
    unique_indices = np.unique(temp_indices.flatten())
    v2d = fn.vertex_to_dof_map(V)
    dof_indices = v2d[unique_indices]
    return dof_indices
    
dof_indices = get_boundaries(V, boundaries)
print('Number of boundary dofs: ', len(dof_indices))

In [6]:
# Setting:
kappa = 200.
kappa_expr = fn.Constant(kappa)

# Define simulation times:
T = 1.0
dt = 0.005
simulation_times = np.arange(0., 1. + 0.5 * dt, dt)

# Define mass matrix:
u_trial = fn.TrialFunction(V)
u_test = fn.TestFunction(V)
M_matrix = fn.assemble(fn.inner(u_trial, u_test) * fn.dx)

In [7]:
# Initial condition:
ic_expr = fn.Expression('min(5.,exp(-10*(pow(x[0]-0.1,2) +  pow(x[1]+0.1,2))))', element=V.ufl_element())
u_init = fn.interpolate(ic_expr, V).vector()
# Boundary condition:
bc = fn.DirichletBC(V, 0., "on_boundary")
bc.apply(u_init)
# Source term:
sigma = 0.01
f_true_time_func = lambda t: 1/(np.sqrt(2*np.pi*sigma**2)) * np.exp(-(t - 0.02)**2 /(2 * sigma**2)) * (t - 0.02) * 1 / sigma**2
f_true_time = f_true_time_func(simulation_times)
f_true = space_time_mult(f_true_time, u_init, M_matrix, simulation_times)
# Zero vectors:
zero_vec = init_vector(M_matrix, 0)
zero_time_vec = init_time_vector(M_matrix, simulation_times, 0)

In [8]:
# Check adjoint operator with end time observation:
parameter = [kappa_expr, 0.1, 0.8]
# u_time, u_dot_time = FractionalWaveSolverNewmark(V, simulation_times, [zero_vec, zero_vec, f_true], 
#                                                  bc, parameter, gamma=0.5, beta=0.25)
# u_obs = u_time[simulation_times[-1]]
# v_temp = init_vector(M_matrix, 0)
# v_temp[:] = np.random.normal(0, 1, V.dim())
# p_time, p_dot_time = FractionalWaveSolverNewmarkAdjoint(V, simulation_times, [zero_vec, v_temp, zero_time_vec], 
#                                                         bc, parameter, gamma=0.5, beta=0.25)
# v_adj = p_dot_time[simulation_times[0]]
# print('Compute <G[u], v>   : ', u_obs.inner(M_matrix * v_temp))
# print('Compute <u, G^*[v]> : ', u_init.inner(M_matrix * v_adj))

In [9]:
from tutorial_PAT_integration import *
# Define misfit:
observation_times = np.arange(0.5, 1. + 0.5 * dt, dt)
print('Observation times:   ', observation_times[[0, 1, 2, -3, -2, -1]])
misfit = NewWaveObservation(V, simulation_times, observation_times, B_matrix)
u_time, _ = FractionalWaveSolverNewmark(V, simulation_times, [zero_vec, zero_vec, f_true], bc, parameter, gamma=0.5, beta=0.25)
# Compute observation:
p_obs = misfit.observe(u_time)
misfit.p_obs = p_obs
rel_noise = 0.01
MAX = p_obs.norm("linf", "linf")
noise_std_dev = rel_noise * MAX
parRandom.normal_perturb(noise_std_dev, p_obs)
misfit.noise_variance = noise_std_dev ** 2
print('Noise variance:      ', misfit.noise_variance)

In [10]:
class GaussianPrior(Prior):
    def __init__(self, Vh, covariance, mean=None):
        """
        Constructor

        Inputs:
        - :code:`Vh`:             Finite element space on which the prior is
                                  defined. Must be the Real space with one global 
                                  degree of freedom
        - :code:`covariance`:     The covariance of the prior. Must be a
                                  :code:`numpy.ndarray` of appropriate size
        - :code:`mean`(optional): Mean of the prior distribution. Must be of
                                  type `dolfin.Vector()`
        """

        self.Vh = Vh
        rel_tol = 1e-12
        max_iter = 100
    
        if Vh.dim() != covariance.shape[0] or Vh.dim() != covariance.shape[1]:
            raise ValueError("Covariance incompatible with Finite Element space")

        if not np.issubdtype(covariance.dtype, np.floating):
            raise TypeError("Covariance matrix must be a float array")

        
        trial = fn.TrialFunction(Vh)
        test  = fn.TestFunction(Vh)
        
        varfM = fn.inner(trial, test) * fn.dx
         
        self.M = fn.assemble(varfM)
        covariance_discrete = self.M.array() @ covariance
        # Define R:
        temp_R = PETSc.Mat()
        temp_R.createDense(covariance.shape, array=covariance_discrete)
        temp_R.assemble()
        R = fn.PETScMatrix(temp_R)
        self.R = fn.as_backend_type(R)
        
        # Define sqrtR:
        sqrt_covariance = sqrtm(covariance)
        temp_sqrtR = PETSc.Mat()
        temp_sqrtR.createDense(sqrt_covariance.shape, array=sqrt_covariance)
        temp_sqrtR.assemble()
        self.sqrtR = fn.PETScMatrix(temp_sqrtR)
        
        self.Rsolver = fn.PETScKrylovSolver("cg")
        self.Rsolver.set_operator(self.M)
        self.Rsolver.parameters["maximum_iterations"] = max_iter
        self.Rsolver.parameters["relative_tolerance"] = rel_tol
        self.Rsolver.parameters["error_on_nonconvergence"] = True
        self.Rsolver.parameters["nonzero_initial_guess"] = False
        
        self.Msolver = PETScKrylovSolver(self.Vh.mesh().mpi_comm(), "cg", "jacobi")
        self.Msolver.set_operator(self.M)
        self.Msolver.parameters["maximum_iterations"] = max_iter
        self.Msolver.parameters["relative_tolerance"] = rel_tol
        self.Msolver.parameters["error_on_nonconvergence"] = True
        self.Msolver.parameters["nonzero_initial_guess"] = False
        
        if mean:
            self.mean = mean
        else:
            tmp = fn.Vector()
            self.M.init_vector(tmp, 0)
            tmp.zero()
            self.mean = tmp
        
    def init_vector(self, x, dim):
        """
        Inizialize a vector :code:`x` to be compatible with the 
        range/domain of :math:`R`.

        If :code:`dim == "noise"` inizialize :code:`x` to be compatible 
        with the size of white noise used for sampling.
        """

        if dim == "noise":
            # self.sqrtRinv.init_vector(x, 1)
            self.sqrtR.init_vector(x, 1)
        else:
            # self.sqrtRinv.init_vector(x, dim)
            self.sqrtR.init_vector(x, dim)

    def sample(self, noise, s, add_mean=True):
        """
        Given :code:`noise` :math:`\\sim \\mathcal{N}(0, I)` compute a 
        sample :code:`s` from the prior.

        If :code:`add_mean == True` add the prior mean value to :code:`s`.
        """
       
        self.sqrtRinv.mult(noise, s)

        if add_mean:
            s.axpy(1.0, self.mean)

In [11]:
# Define prior:
gamma = 1.
delta = 10.
prior_2 = BiLaplacianPrior(V, gamma, delta)
prior = BiLaplacianPrior(V, 0.01, 10.)
covariance = np.eye(V.dim())
# prior = GaussianPrior(V, covariance)
prior.mean = fn.interpolate(fn.Constant(0.25), V).vector()

In [12]:
WaveProblem = WaveInverse(V, simulation_times, misfit, prior)
X = WaveProblem.generate_vector()
X[PARAMETER][:] = u_init[:]
WaveProblem.solveFwd(X[STATE], X)

In [13]:
# State variables:
objs = [fn.Function(V, X[STATE][0.5]), fn.Function(V, X[STATE][0.6]), fn.Function(V, X[STATE][1.])]
mytitles = ["t = 0.5", "t = 0.6", "t = 1"]
nb.multi1_plot(objs, mytitles, same_colorbar=True, cmap='coolwarm')
plt.show()

In [14]:
# Observation:
u_obs_1 = init_vector(M_matrix, 0)
u_obs_1[misfit.targets] = p_obs[0.5][misfit.targets]

u_obs_2 = init_vector(M_matrix, 0)
u_obs_2[misfit.targets] = p_obs[0.6][misfit.targets]

u_obs_3 = init_vector(M_matrix, 0)
u_obs_3[misfit.targets] = p_obs[1][misfit.targets]

objs = [fn.Function(V, u_obs_1), fn.Function(V, u_obs_2), fn.Function(V, u_obs_3)]
mytitles = ["t = 0.5", "t = 0.6", "t = 1"]
nb.multi1_plot(objs, mytitles, same_colorbar=True, cmap='coolwarm')
plt.show()

In [15]:
_cost, reg, _misfit = WaveProblem.cost(X)
print('Regularization term: ', reg)
print('Misfit term:         ', _misfit)
print('Cost:                ', _cost)

In [16]:
X_2 = WaveProblem.generate_vector()
dm = init_vector(M_matrix, 0)
dm[:] = 100 * np.random.normal(0, 1, V.dim())
bc.apply(dm)
grad_misfit_1, grad_reg_1 = WaveProblem.evalGradientFD(X_2, dm, 1e-8)
grad_misfit_2, grad_reg_2 = WaveProblem.evalGradientAD(X_2, dm)
grad_misfit_3, grad_reg_3 = WaveProblem.evalGradientADForward(X_2, dm)

In [17]:
print('Gradient misfit: ', grad_misfit_1, grad_reg_1)
print('Gradient misfit: ', grad_misfit_2, grad_reg_2)
print('Gradient misfit: ', grad_misfit_3, grad_reg_3)

In [ ]:
X_iter = WaveProblem.generate_vector()
X_iter[PARAMETER][:] = 1.

_cost, reg, _misfit = WaveProblem.cost(X_iter)
WaveProblem.solveParameter(X_iter, preconditioner=prior_2.Rsolver)

In [126]:
u_func = fn.Function(V, X_iter[PARAMETER])
c = fn.plot(u_func, cmap='coolwarm')
plt.colorbar(c)
plt.show()

In [127]:
u_func = fn.Function(V, u_init)
c = fn.plot(u_func, cmap='coolwarm')
plt.colorbar(c)
plt.show()

In [ ]:
objs = [fn.Function(V, u_init), fn.Function(V, X_iter[PARAMETER])]
mytitles = ["Ground truth", "Reconstruction", "t = 1"]
nb.multi1_plot(objs, mytitles, same_colorbar=True, cmap='coolwarm')
plt.show()

In [ ]:
# Print new cost:
_cost, reg, _misfit = WaveProblem.cost(X_iter)
print('Regularization term: ', reg)
print('Misfit term:         ', _misfit)
print('Cost:                ', _cost)

In [ ]:
prior = LaplacianPrior(V, 0.01, 1.)

In [ ]:
u = init_vector(M_matrix, 0)
u[:] = 1.
v = init_vector(M_matrix, 0)
prior.R.mult(u, v)
print(v[:])

In [ ]:
m_vector = init_vector(M_matrix, 0)
m_index = init_vector(M_matrix, 0)
grid = V.tabulate_dof_coordinates()
center_circle = np.array([0.3, 0.1])
center_rectangular = np.array([-0.3, 0.1])
for i in range(V.dim()):
    m_index[i] = np.linalg.norm(grid[i] - center_circle) ** 2 < 0.3 ** 2